### 주제: 건설 장비 내부 기계 부품의 마모 상태 및 윤활 성능을 오일 데이터 분석을 통해 확인하고, AI를 활용한 분류 모델 개발을 통해 적절한 교체 주기를 파악

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

샘플 오일 관련 부품 정보를 분석해 오일 정상 여부 예측

In [10]:
train = pd.read_csv('D:/김동영/11_Github/mygit-1/데이터사이언스수업/과제/최종과제/HD현대해커톤/train.csv')
test_df = pd.read_csv('D:/김동영/11_Github/mygit-1/데이터사이언스수업/과제/최종과제/HD현대해커톤/test.csv')
submission = pd.read_csv('D:/김동영/11_Github/mygit-1/데이터사이언스수업/과제/최종과제/HD현대해커톤/sample_submission.csv')

In [11]:
test_df.columns

Index(['ID', 'COMPONENT_ARBITRARY', 'ANONYMOUS_1', 'YEAR', 'ANONYMOUS_2', 'AG',
       'CO', 'CR', 'CU', 'FE', 'H2O', 'MN', 'MO', 'NI', 'PQINDEX', 'TI', 'V',
       'V40', 'ZN'],
      dtype='object')

In [12]:
columns = ['ID', 'COMPONENT_ARBITRARY', 'ANONYMOUS_1', 'YEAR', 'ANONYMOUS_2', 'AG',
       'CO', 'CR', 'CU', 'FE', 'H2O', 'MN', 'MO', 'NI', 'PQINDEX', 'TI', 'V',
       'V40', 'ZN','Y_LABEL']

train 데이터에는 존재하나 test 데이터에는 없는 변수들 존재했었다. 이는 모델 학습 시 사용할 수 없는 변수들이기 때문에 제거해주었다.    

In [13]:
train_df = train[columns]

In [15]:
train_df['COMPONENT_ARBITRARY'].value_counts()

COMPONENT_ARBITRARY
COMPONENT3    7050
COMPONENT1    3890
COMPONENT2    2316
COMPONENT4     839
Name: count, dtype: int64

부품 종류로는 4개가 있으며 1은 3890개, 2는 2316개, 3은 7050개, 4는 839개씩 있다.    

In [16]:
# COMPONENT_ARBITRARY별 Y_LABEL의 0,1 개수
count = train_df.groupby('COMPONENT_ARBITRARY')['Y_LABEL'].value_counts().unstack().fillna(0)
print("개수:\n", count)

# COMPONENT_ARBITRARY별 Y_LABEL의 0,1 비율
ratio = train_df.groupby('COMPONENT_ARBITRARY')['Y_LABEL'].value_counts(normalize=True).unstack().fillna(0)
print("\n비율:\n", ratio)


개수:
 Y_LABEL                 0     1
COMPONENT_ARBITRARY            
COMPONENT1           3500   390
COMPONENT2           1915   401
COMPONENT3           4906  2144
COMPONENT4            750    89

비율:
 Y_LABEL                     0         1
COMPONENT_ARBITRARY                    
COMPONENT1           0.899743  0.100257
COMPONENT2           0.826857  0.173143
COMPONENT3           0.695887  0.304113
COMPONENT4           0.893921  0.106079


COMPONENT_ARBITRARY별로 오일 정상, 비정상 개수 및 비율을 확인해본 결과 COMPONENT 2와 3에서 이상 오일 비율이 높게 나온 것을 확인할 수 있었다.

변수 정보   
ANONYMOUS_1 - 무명 Feature 1. 수치형 데이터   
ANONYMOUS_2 - 무명 Feature 2. 수치형 데이터   
AG, CO, CR, FE, H2O, MN, MO, NI, TI, V, ZN - 원소기호   
V40 - 40도에서 측정한 액체의 점도   
YEAR - 오일샘플 및 진단 해(Year)    
PQINDEX(Particle Quantifier Index) : Particle Quantifier Index(PQ Index), 또는 PQI라고도 불리는 이 지표는 입자의 크기와 상관없이 윤활유 샘플 내에 존재하는 전체 강자성 물질(자성을 띤 금속 입자)의 양을 측정하는 방법이다.
이 지표는 마모로 인한 문제가 발생할 가능성을 조기에 감지하기 위한 선별 검사 도구로 사용되며, 예지 보전(predictive maintenance) 프로그램에서 자주 활용된다.    


In [17]:
train_df.groupby('COMPONENT_ARBITRARY').mean(numeric_only=True)

# 평균값을 표로 보기 좋게 출력
mean_df = train_df.groupby('COMPONENT_ARBITRARY').mean(numeric_only=True)
display(mean_df)

,ANONYMOUS_1,YEAR,ANONYMOUS_2,AG,CO,CR,CU,FE,H2O,MN,MO,NI,PQINDEX,TI,V,V40,ZN,Y_LABEL
COMPONENT_ARBITRARY,,,,,,,,,,,,,,,,,,
COMPONENT1,-0.986054,2013.729563,-0.959173,0.025193,0.014396,1.910540,9.929563,27.057069,0.002699,0.514910,69.364267,0.152185,20.422622,0.055013,0.024422,102.678103,1161.010283,0.100257
COMPONENT2,-0.986206,2013.718912,-0.961256,0.024180,0.009499,0.515976,56.677029,18.890328,0.003282,0.155872,0.652418,0.027634,27.230570,0.032815,0.010794,52.720695,505.785838,0.173143
COMPONENT3,-0.984818,2013.494043,-0.961531,0.028227,0.044397,5.361986,32.252057,332.500426,0.064525,5.137163,3.415745,1.362553,807.300567,1.372908,0.082837,136.411370,241.379574,0.304113
COMPONENT4,-0.984588,2014.443385,-0.953130,0.013111,0.007151,0.091776,107.109654,21.651967,0.001549,0.852205,50.958284,0.034565,21.091776,0.010727,0.011919,69.310012,1081.669845,0.106079


Component 3에서 이상 샘플과 정상 샘플의 수치를 비교해보았을 때 ANONYMOUS_1, ANONYMOUS_2, YEAR, AG 변수를 제외한 나머지 변수들에서 수치적으로 큰 차이를 보였다.    

In [18]:
# COMPONENT3만 추출
comp3_df = train_df[train_df['COMPONENT_ARBITRARY'] == 'COMPONENT3']

# Y_LABEL이 0과 1인 데이터 각각 추출
comp3_0 = comp3_df[comp3_df['Y_LABEL'] == 0]
comp3_1 = comp3_df[comp3_df['Y_LABEL'] == 1]

# 주요 변수별 평균 비교
mean_0 = comp3_0.mean(numeric_only=True)
mean_1 = comp3_1.mean(numeric_only=True)
diff = pd.DataFrame({'Y_LABEL=0': mean_0, 'Y_LABEL=1': mean_1})

display(diff)

,Y_LABEL=0,Y_LABEL=1
ANONYMOUS_1,-0.984669,-0.985158
YEAR,2013.614757,2013.217817
ANONYMOUS_2,-0.962026,-0.960398
AG,0.024868,0.035914
CO,0.023848,0.091418
CR,1.779250,13.560168
CU,19.878516,60.565765
FE,156.025275,736.319030
H2O,0.004260,0.202425
MN,2.804525,10.474813


In [19]:
from scipy.stats import ttest_ind, mannwhitneyu

In [20]:
# COMPONENT3 데이터에서 독립변수 리스트 (Y_LABEL, ID, COMPONENT_ARBITRARY 제외)
independent_vars = [col for col in comp3_df.columns if col not in ['ID', 'COMPONENT_ARBITRARY', 'Y_LABEL']]

# 결과 저장용
test_results = []

for col in independent_vars:
  # 두 그룹 데이터
  group0 = comp3_0[col]
  group1 = comp3_1[col]
  # 정규성, 등분산성 가정이 어려우므로 Mann-Whitney U test 사용
  stat, p = mannwhitneyu(group0, group1, alternative='two-sided')
  test_results.append({'변수': col, 'Mann-Whitney U 통계량': stat, 'p-value': p})

# 결과 DataFrame으로 정리
test_df = pd.DataFrame(test_results)
display(test_df.sort_values('p-value'))

,변수,Mann-Whitney U 통계량,p-value
5,CR,2238235.5,0.000000e+00
7,FE,1886591.0,0.000000e+00
12,PQINDEX,1545047.0,0.000000e+00
9,MN,2350153.5,8.060757e-308
13,TI,3475027.5,2.531391e-278
11,NI,3286839.5,1.446952e-187
8,H2O,4719262.5,7.578654e-77
14,V,4898603.0,2.452805e-41
4,CO,5041572.5,5.371950e-19
6,CU,4564088.0,5.470530e-19


통계적으로 독립 변수와 종속 변수간의 유의미성을 확인하고자  Mann-Whitney U 검정을 진행했다.   
CR, FE, PQINDEX, MN, TI, NI, H2O, V, CO, CU, MO, V40, YEAR, AG 등은 오일의 정상/비정상 여부(Y_LABEL)와 강한 관련성이 있는 반면, ANONYMOUS_1, ZN, ANONYMOUS_2는 정상/비정상 여부와 관련성이 약하거나 없었다.   

Mann-Whitney U 검정을 사용한 이유   
t-test 검정은 두 그룹의 평균 차이를 검정하는 방법인데,
**정규성(각 그룹의 데이터가 정규분포를 따른다는 가정)**과 **등분산성(두 그룹의 분산이 비슷하다는 가정)**이 필요합니다.

하지만 실제 오일 데이터(특히 원소 농도, PQINDEX 등)는

정규분포를 따르지 않는 경우가 많고
이상치(outlier)가 많거나 분포가 치우쳐 있는 경우가 많습니다.
이럴 때 t-test를 사용하면 결과가 왜곡될 수 있기 때문에
비모수 검정인 Mann-Whitney U test(두 그룹의 분포 차이를 비교, 정규성 가정 불필요)를 사용하는 것이 더 적합